In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
!pip install -q sodapy pandas

In [ ]:
from sodapy import Socrata
import pandas as pd
import os
import time
from requests.exceptions import ReadTimeout

# Connect to Chicago Open Data
client = Socrata(
    "data.cityofchicago.org",
    None,
    timeout=60
)

# Configuration
LIMIT = 100000          # 1 lakh rows per request
MAX_ROWS = 1000000      # Total 10 lakh rows
offset = 0

SAVE_FOLDER = "/content/chicago_chunks"
os.makedirs(SAVE_FOLDER, exist_ok=True)

print("Download Started...\n")

while offset < MAX_ROWS:
    try:
        print(f"Downloading rows {offset:,} to {min(offset + LIMIT, MAX_ROWS):,}...")

        results = client.get(
            "ijzp-q8t2",
            limit=LIMIT,
            offset=offset
        )

        if not results:
            print("No more data available.")
            break

        df = pd.DataFrame(results)

        file_name = f"{SAVE_FOLDER}/crime_{offset}.csv"
        df.to_csv(file_name, index=False)

        print(f"✅ Saved {len(df)} rows -> {file_name}")

        offset += LIMIT

        time.sleep(2)

    except ReadTimeout:
        print("⚠️ Timeout! Retrying after 10 seconds...")
        time.sleep(10)

print("\n🎉 Download Completed!")
print(f"Total Rows Downloaded: {offset:,}")

In [ ]:
import pandas as pd
import glob

files = sorted(glob.glob("/content/chicago_chunks/*.csv"))

df = pd.concat(
    (pd.read_csv(file, low_memory=False) for file in files),
    ignore_index=True
)

print(df.shape)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
df.duplicated().sum()

In [ ]:
df['date'] = pd.to_datetime(df['date'])

In [ ]:
df['year'] = pd.to_numeric(df['year'])

In [ ]:
df['latitude'] = pd.to_numeric(df['latitude'])

In [ ]:
df['longitude'] = pd.to_numeric(df['longitude'])

In [ ]:
df['x_coordinate'] = pd.to_numeric(df['x_coordinate'])

In [ ]:
df['y_coordinate'] = pd.to_numeric(df['y_coordinate'])

In [ ]:
df = df.drop(columns=[
    ':@computed_region_awaf_s7ux',
    ':@computed_region_6mkv_f3dw',
    ':@computed_region_vrxf_vc4k',
    ':@computed_region_bdys_3d7i',
    ':@computed_region_43wa_7qmu',
    ':@computed_region_rpca_8um6',
    ':@computed_region_d9mm_jgwp',
    ':@computed_region_d3ds_rm58',
    ':@computed_region_8hcu_yrd4'
])

In [ ]:
df['location_description'] = df['location_description'].fillna('Unknown')

In [ ]:
df['ward'] = df['ward'].fillna(df['ward'].mode()[0])

In [ ]:
df['community_area'] = df['community_area'].fillna(df['community_area'].mode()[0])

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['hour'] = df['date'].dt.hour

In [ ]:
df['day_name'] = df['date'].dt.day_name()

In [ ]:
df['month'] = df['date'].dt.month_name()

In [ ]:
def time_slot(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df['time_slot'] = df['hour'].apply(time_slot)

In [ ]:
df[['date', 'hour', 'time_slot']].head(10)

In [ ]:
district_time = pd.crosstab(df['district'], df['time_slot'])

district_time

In [ ]:

district_names = {
    1: "Central",
    2: "Wentworth",
    3: "Grand Crossing",
    4: "South Chicago",
    5: "Calumet",
    6: "Gresham",
    7: "Englewood",
    8: "Chicago Lawn",
    9: "Deering",
    10: "Ogden",
    11: "Harrison",
    12: "Near West",
    14: "Shakespeare",
    15: "Austin",
    16: "Jefferson Park",
    17: "Albany Park",
    18: "Near North",
    19: "Town Hall",
    20: "Lincoln",
    22: "Morgan Park",
    24: "Rogers Park",
    25: "Grand Central",
    31: "Unknown"
}

In [ ]:
df['district_name'] = df['district'].map(district_names)

In [ ]:
df[['district', 'district_name']].head()

In [ ]:
district_time = pd.crosstab(df['district_name'], df['time_slot'])

district_time

In [ ]:
plt.figure(figsize=(12,8))

sns.heatmap(
    district_time,
    annot=True,
    fmt='d',
    cmap='Reds'
)

plt.title("Crime Count by District and Time Slot")
plt.xlabel("Time Slot")
plt.ylabel("District")
plt.show()

In [ ]:
crime_type = pd.crosstab(
    [df['district_name'], df['time_slot']],
    df['primary_type']
)

crime_type

In [ ]:
def most_common_crime(district, time_slot):
    
    data = df[
        (df['district_name'] == district) &
        (df['time_slot'] == time_slot)
    ]

    if data.empty:
        print("No data available.")
    else:
        crime = data['primary_type'].mode()[0]
        total_crimes = len(data)

        print("District :", district)
        print("Time Slot :", time_slot)
        print("Most Common Crime :", crime)
        print("Total Crimes :", total_crimes)

In [ ]:
most_common_crime("Chicago Lawn", "Night")

In [ ]:
def crime_risk_analysis(district, time_slot):

    data = df[
        (df['district_name'] == district) &
        (df['time_slot'] == time_slot)
    ]

    if data.empty:
        print("No data available for this district and time.")
        return

    total_crimes = len(data)
    common_crime = data['primary_type'].mode()[0]

    # Risk Level
    if total_crimes >= 18000:
        risk = "🔴 HIGH"
    elif total_crimes >= 12000:
        risk = "🟡 MEDIUM"
    else:
        risk = "🟢 LOW"

    print("=" * 50)
    print("Crime Risk Analysis")
    print("=" * 50)

    print("District        :", district)
    print("Time Slot       :", time_slot)
    print("Total Crimes    :", total_crimes)
    print("Most Common Crime :", common_crime)
    print("Risk Level      :", risk)

    print("\nSafety Tips")

    if risk == "🔴 HIGH":
        print("• Avoid isolated streets.")
        print("• Travel with someone if possible.")
        print("• Share your live location.")
        print("• Prefer trusted cab or public transport.")
        print("• Stay alert and avoid carrying valuables.")

    elif risk == "🟡 MEDIUM":
        print("• Stay alert.")
        print("• Use well-lit roads.")
        print("• Keep your phone charged.")
        print("• Inform a family member about your trip.")

    else:
        print("• Area appears comparatively safer based on historical data.")
        print("• Stay aware of your surroundings.")
        print("• Follow normal safety precautions.")

In [ ]:
crime_risk_analysis("Chicago Lawn", "Night")

In [ ]:
district = input("Enter District Name : ")
time_slot = input("Enter Time Slot (Morning/Afternoon/Evening/Night): ")

crime_risk_analysis(district, time_slot)

In [ ]:
plt.figure(figsize=(12,6))

sns.countplot(
    data=df,
    y='primary_type',
    order=df['primary_type'].value_counts().head(10).index
)

plt.title("Top 10 Crime Types")
plt.xlabel("Crime Count")
plt.ylabel("Crime Type")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    x='time_slot',
    order=['Morning','Afternoon','Evening','Night']
)

plt.title("Crime Count by Time Slot")
plt.xlabel("Time Slot")
plt.ylabel("Crime Count")
plt.show()

In [ ]:
plt.figure(figsize=(12,8))

sns.countplot(
    data=df,
    y='district_name',
    order=df['district_name'].value_counts().index
)

plt.title("Crime Count by District")
plt.xlabel("Crime Count")
plt.ylabel("District")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    data=df,
    x='arrest'
)

plt.title("Arrest Analysis")
plt.xlabel("Arrest")
plt.ylabel("Crime Count")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    data=df,
    x='domestic'
)

plt.title("Domestic Crime Analysis")
plt.xlabel("Domestic Crime")
plt.ylabel("Crime Count")
plt.show()

In [ ]:
district_time = pd.crosstab(df['district_name'], df['time_slot'])

plt.figure(figsize=(12,8))

sns.heatmap(
    district_time,
    annot=True,
    fmt='d',
    cmap='Reds'
)

plt.title("Crime Count by District and Time Slot")
plt.show()

In [1]:
risk_score = df.groupby("district_name").size().reset_index(name="crime_count")

risk_score["Risk_Level"] = pd.cut(
    risk_score["crime_count"],
    bins=3,
    labels=["Low", "Medium", "High"]
)

risk_score

NameError: name 'df' is not defined

In [ ]:
top10 = df['district_name'].value_counts().head(10)

plt.figure(figsize=(10,6))
top10.plot(kind='bar')
plt.title("Top 10 Dangerous Districts")
plt.ylabel("Crime Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
yearly = df.groupby("year").size()

plt.figure(figsize=(10,5))
yearly.plot(marker='o')
plt.title("Crime Trend Over Years")
plt.ylabel("Crime Count")
plt.show()

In [ ]:
df["weekend"] = df["day_name"].isin(["Saturday","Sunday"])

week = df["weekend"].value_counts()

plt.figure(figsize=(5,5))
week.plot(kind="pie",autopct="%1.1f%%")
plt.ylabel("")
plt.title("Weekend vs Weekday Crime")
plt.show()